# Gaussian: can label smoothing beat ordinary CE on clean test CE?

## 1. Question and comparison rule

Sweep training-label flip probability **eta** and binary label-smoothing strength **epsilon** for the same Gaussian logistic model and paired seeds as notebook 03.

Smoothing uses
$$
y_\varepsilon=(1-\varepsilon)y+\varepsilon/2,\qquad
L_{\mathrm{LS}}(z,y;\varepsilon)=\operatorname{softplus}(z)-y_\varepsilon z.
$$
At every eta, compare smoothing against ordinary CE trained with the **same corrupted labels, data, initialization, optimizer, and epoch budget**. Select learning rate separately for the baseline and each smoothing strength using clean validation CE and three tuning seeds. Freeze those rates before evaluation on five matched seeds.

The primary score is
$$
\mathrm{gain}(\eta,\varepsilon)
=\frac{1}{S}\sum_s\left[
\mathrm{CE}_{\mathrm{test}}(\mathrm{CE},\eta,s)
-\mathrm{CE}_{\mathrm{test}}(\mathrm{LS},\eta,\varepsilon,s)
\right].
$$
Positive gain means lower ordinary clean test CE after smoothing. The heatmap uses a continuous red-white-green scale centered at zero: red for worse CE, green for better CE, and white for zero or negligible change. Saturation reflects absolute gain magnitude.

The epsilon grid equals twice the delta grid from notebook 03. This matches the positive-margin penalty when epsilon = 2 * delta, while preserving the difference on negative margins. No radius or smoothing strength is selected using test results.

The notebook runs independently on CPU with uv. Compatible CE-baseline records from notebook 03 may be reused; missing or incompatible baselines are trained locally. All text is in English, figures use Plotly, and no tables are displayed.


In [1]:
import ast
import csv
import hashlib
import json
import statistics
import time
from itertools import product
from pathlib import Path

import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "pyproject.toml").exists()
RESULTS_ROOT = PROJECT_ROOT / "results" / "04_gaussian_eta_epsilon"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
DTYPE = torch.float32
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

ETAS = (0.0, 0.025, 0.05, 0.1, 0.2, 0.3, 0.4)
EPSILONS = (0.0, 0.0002, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.8, 1.0)
LEARNING_RATES = (0.001, 0.01, 0.05)
TUNING_SEEDS = (0, 1, 2)
EVALUATION_SEEDS = (201, 202, 203, 204, 205)
SPLIT_SIZES = {"train": 1000, "validation": 400, "test": 2000}
EPOCHS = 400
WIN_ATOL = 1e-6
REUSE_CACHE = True
REUSE_CE_BASELINE = True

assert set(TUNING_SEEDS).isdisjoint(EVALUATION_SEEDS)
assert ETAS == tuple(sorted(set(ETAS))) and EPSILONS == tuple(sorted(set(EPSILONS)))
assert all(0 <= eta < 0.5 for eta in ETAS)
assert all(0 <= epsilon <= 1 for epsilon in EPSILONS) and 0.0 in EPSILONS
assert all(size % 2 == 0 for size in SPLIT_SIZES.values())
TOTAL_TUNING_RUNS = len(ETAS) * len(EPSILONS) * len(LEARNING_RATES) * len(TUNING_SEEDS)
TOTAL_EVALUATION_RUNS = len(ETAS) * len(EPSILONS) * len(EVALUATION_SEEDS)

print(f"PyTorch {torch.__version__} | {DEVICE} | {DTYPE}")
print(f"Grid: {len(ETAS)} eta values x {len(EPSILONS)} epsilon values.")
print(f"Tuning: {TOTAL_TUNING_RUNS} unique runs; evaluation: {TOTAL_EVALUATION_RUNS} unique runs.")
print(f"Win: mean paired clean-test CE gain > {WIN_ATOL:g}.")


PyTorch 2.11.0+cpu | cpu | torch.float32
Grid: 7 eta values x 12 epsilon values.
Tuning: 756 unique runs; evaluation: 420 unique runs.
Win: mean paired clean-test CE gain > 1e-06.


## 2. Matched data and training

Generate balanced Gaussian classes with
$$
x\mid y\sim\mathcal N((2y-1)(1,1)^\top,I_2),\qquad y\in\{0,1\}.
$$
Independently generate train, validation, and test splits. Fit feature standardization only on training features.

Seeds vary datasets and initial model weights. Reuse both across eta/epsilon/lr combinations. Draw one corruption-uniform vector per seed and flip training label i iff its uniform draw is below eta. All strengths and the CE baseline share identical observed labels; masks are nested across eta.

Use a 3-parameter logistic model and 400 full-batch Adam steps. Epsilon = 0 is exactly ordinary CE. Epsilon = 1 makes both observed labels map to target 0.5. The baseline is trained or reused once per eta/seed/lr, not once for every positive epsilon.


In [2]:
def generator(seed):
    return torch.Generator(device="cpu").manual_seed(seed)


def gaussian_split(n, seed):
    g = generator(seed)
    labels = torch.cat((torch.zeros(n // 2), torch.ones(n // 2))).to(DTYPE)
    features = torch.randn(n, 2, generator=g, dtype=DTYPE) + (2 * labels - 1).unsqueeze(1)
    permutation = torch.randperm(n, generator=g)
    return features[permutation].to(DEVICE), labels[permutation].to(DEVICE)


def prepare_data(seed, include_test=False):
    splits = ("train", "validation", "test") if include_test else ("train", "validation")
    data = {
        split: gaussian_split(SPLIT_SIZES[split], seed * 1000 + split_id)
        for split_id, split in enumerate(splits)
    }
    mean = data["train"][0].mean(dim=0)
    scale = data["train"][0].std(dim=0, unbiased=False)
    assert torch.all(scale > 0)
    for split, (features, labels) in data.items():
        data[split] = ((features - mean) / scale, labels)
    data["corruption_uniforms"] = torch.rand(
        SPLIT_SIZES["train"], generator=generator(seed * 1000 + 999), dtype=DTYPE
    )
    return data


def observed_labels(data, eta):
    clean = data["train"][1]
    observed = torch.where(data["corruption_uniforms"] < eta, 1 - clean, clean)
    realized = (observed != clean).to(DTYPE).mean().item()
    return observed, realized


def make_model(seed):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed + 10_000)
        model = nn.Linear(2, 1)
    return model.to(device=DEVICE, dtype=DTYPE)


assert sum(p.numel() for p in make_model(0).parameters()) == 3


In [3]:
def objective(logits, labels, epsilon):
    ce = F.softplus(logits) - labels * logits
    if epsilon == 0:
        return ce.mean()
    soft_labels = (1 - epsilon) * labels + epsilon / 2
    return (F.softplus(logits) - soft_labels * logits).mean()


@torch.no_grad()
def evaluate(model, split):
    features, labels = split
    logits = model(features).squeeze(-1)
    if not torch.isfinite(logits).all():
        raise FloatingPointError("Nonfinite predictions.")
    return {
        "ce": F.binary_cross_entropy_with_logits(logits, labels).item(),
        "accuracy": ((logits >= 0) == labels.bool()).to(DTYPE).mean().item(),
        "confidence": torch.sigmoid(logits.abs()).mean().item(),
    }


def train_one(seed, data, observed, epsilon, lr):
    model = make_model(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    features = data["train"][0]
    started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        optimizer.zero_grad(set_to_none=True)
        loss = objective(model(features).squeeze(-1), observed, epsilon)
        if epoch % 20 == 0 and not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss at epoch {epoch}.")
        loss.backward()
        optimizer.step()
    seconds = time.perf_counter() - started
    model.eval()
    return model, evaluate(model, data["validation"]), seconds


def summarize(values):
    return statistics.mean(values), statistics.stdev(values) if len(values) > 1 else 0.0


def write_csv(path, records):
    if not records:
        return
    fields = list(dict.fromkeys(key for row in records for key in row))
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def read_csv(path):
    records = []
    with path.open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            parsed = {}
            for key, value in row.items():
                if key in ("status", "error"):
                    parsed[key] = value
                elif value == "":
                    parsed[key] = None
                elif key == "seed":
                    parsed[key] = int(value)
                else:
                    parsed[key] = float(value)
            records.append(parsed)
    return records


def run_key(record):
    return record["seed"], record["eta"], record["epsilon"], record["lr"]


### Reproducibility, cache, and CE-baseline reuse

Successful run records are cached under a fingerprint of numerical settings, PyTorch version, and exact data/training sources. Plot-only edits reuse completed records.

If notebook 03 has compatible artifacts, reuse only its delta = 0 CE records as epsilon = 0 baselines. Check numerical settings, data source, train/evaluation function structure, and the zero-strength CE branch before reuse. Otherwise train the baseline locally. No positive-delta RobustCE records are imported.

Failed runs are retried on later executions. A tuning candidate needs all three successful seeds. A heatmap cell needs all five valid paired evaluation seeds; incomplete cells are left blank.


In [4]:
settings = {
    "torch_version": str(torch.__version__),
    "device": str(DEVICE), "dtype": str(DTYPE), "threads": torch.get_num_threads(),
    "etas": list(ETAS), "epsilons": list(EPSILONS), "learning_rates": list(LEARNING_RATES),
    "tuning_seeds": list(TUNING_SEEDS), "evaluation_seeds": list(EVALUATION_SEEDS),
    "split_sizes": SPLIT_SIZES, "epochs": EPOCHS, "win_atol": WIN_ATOL,
    "optimizer": "Adam, default betas/eps, no weight decay, no schedule",
    "batching": "full batch", "selection": "mean final ordinary clean validation CE",
}
notebook_path = PROJECT_ROOT / "notebooks" / "04_gaussian_eta_epsilon.ipynb"
notebook_source = json.loads(notebook_path.read_text(encoding="utf-8"))
implementation = "\n".join(
    "".join(cell["source"]) for cell in notebook_source["cells"]
    if cell.get("id") in ("data", "training")
)
fingerprint = hashlib.sha256(
    json.dumps({"settings": settings, "implementation": implementation}, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
RUN_DIR = RESULTS_ROOT / fingerprint
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "settings.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
(RESULTS_ROOT / "latest_run.json").write_text(
    json.dumps({"run_id": fingerprint, "run_directory": str(RUN_DIR)}, indent=2) + "\n", encoding="utf-8"
)
TUNING_PATH = RUN_DIR / "tuning_runs.csv"
EVALUATION_PATH = RUN_DIR / "evaluation_runs.csv"
tuning_cache = {
    run_key(row): row for row in (read_csv(TUNING_PATH) if REUSE_CACHE and TUNING_PATH.exists() else [])
}
evaluation_cache = {
    run_key(row): row for row in (read_csv(EVALUATION_PATH) if REUSE_CACHE and EVALUATION_PATH.exists() else [])
}
print(f"Result directory: {RUN_DIR}")
print(f"Cached records: {len(tuning_cache)} tuning, {len(evaluation_cache)} evaluation.")

# Optional CE-only reuse from notebook 03, guarded by settings and source checks.
def canonical_function(source, name, first_statements=None):
    module = ast.parse(source)
    node = next(
        item for item in module.body
        if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)) and item.name == name
    )

    class NormalizeStrength(ast.NodeTransformer):
        def visit_Name(self, item):
            if item.id in ("delta", "epsilon"):
                item.id = "strength"
            return item

        def visit_arg(self, item):
            if item.arg in ("delta", "epsilon"):
                item.arg = "strength"
            return item

    node = NormalizeStrength().visit(node)
    if first_statements is not None:
        node.body = node.body[:first_statements]
    return ast.dump(node, include_attributes=False)


def compatible_ce_source():
    latest_path = PROJECT_ROOT / "results" / "03_gaussian_eta_delta" / "latest_run.json"
    source_path = PROJECT_ROOT / "notebooks" / "03_gaussian_eta_delta.ipynb"
    if not latest_path.exists() or not source_path.exists():
        return None
    latest = json.loads(latest_path.read_text(encoding="utf-8"))
    directory = Path(latest["run_directory"])
    settings_path = directory / "settings.json"
    if not settings_path.exists():
        return None
    base_settings = json.loads(settings_path.read_text(encoding="utf-8"))
    common_keys = (
        "torch_version", "device", "dtype", "threads", "etas", "learning_rates",
        "tuning_seeds", "evaluation_seeds", "split_sizes", "epochs", "win_atol",
        "optimizer", "batching", "selection",
    )
    if any(base_settings.get(key) != settings[key] for key in common_keys):
        return None
    baseline_notebook = json.loads(source_path.read_text(encoding="utf-8"))
    baseline_cells = {
        cell.get("id"): "".join(cell["source"]) for cell in baseline_notebook["cells"]
    }
    current_cells = {
        cell.get("id"): "".join(cell["source"]) for cell in notebook_source["cells"]
    }
    baseline_implementation = "\n".join(
        "".join(cell["source"]) for cell in baseline_notebook["cells"]
        if cell.get("id") in ("data", "training")
    )
    baseline_fingerprint = hashlib.sha256(
        json.dumps(
            {"settings": base_settings, "implementation": baseline_implementation},
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()[:16]
    if baseline_fingerprint != latest.get("run_id"):
        return None
    if baseline_cells["data"] != current_cells["data"]:
        return None
    for function in ("train_one", "evaluate"):
        if canonical_function(baseline_cells["training"], function) != canonical_function(current_cells["training"], function):
            return None
    if canonical_function(baseline_cells["training"], "objective", 2) != canonical_function(current_cells["training"], "objective", 2):
        return None
    return directory


baseline_source = compatible_ce_source() if REUSE_CACHE and REUSE_CE_BASELINE else None
imported_baseline_tuning = 0
imported_baseline_evaluation = 0
if baseline_source is not None:
    for filename, cache, allowed_seeds, phase in (
        ("tuning_runs.csv", tuning_cache, TUNING_SEEDS, "tuning"),
        ("evaluation_runs.csv", evaluation_cache, EVALUATION_SEEDS, "evaluation"),
    ):
        source_path = baseline_source / filename
        if not source_path.exists():
            continue
        for row in read_csv(source_path):
            if row.get("delta") != 0 or row["status"] != "ok":
                continue
            if row["seed"] not in allowed_seeds or row["eta"] not in ETAS or row["lr"] not in LEARNING_RATES:
                continue
            row = dict(row)
            row["epsilon"] = row.pop("delta")
            key = run_key(row)
            if key in cache and cache[key]["status"] == "ok":
                continue
            cache[key] = row
            if phase == "tuning":
                imported_baseline_tuning += 1
            else:
                imported_baseline_evaluation += 1
    write_csv(TUNING_PATH, list(tuning_cache.values()))
    write_csv(EVALUATION_PATH, list(evaluation_cache.values()))
    print(
        f"Imported compatible CE baselines: {imported_baseline_tuning} tuning, "
        f"{imported_baseline_evaluation} evaluation records."
    )
else:
    print("No compatible external CE baseline imported; missing baselines will be trained locally.")

reuse_manifest_path = RUN_DIR / "baseline_reuse.json"
if not reuse_manifest_path.exists():
    reuse_manifest_path.write_text(
        json.dumps({
            "source_directory": str(baseline_source) if baseline_source is not None else None,
            "imported_tuning_records": imported_baseline_tuning,
            "imported_evaluation_records": imported_baseline_evaluation,
        }, indent=2) + "\n",
        encoding="utf-8",
    )


Result directory: C:\Users\shich\Src\hw1opt\results\04_gaussian_eta_epsilon\5f24bd06b1f030c3
Cached records: 756 tuning, 420 evaluation.
Imported compatible CE baselines: 0 tuning, 0 evaluation records.


## 3. Validation-only learning-rate selection

For every eta and epsilon, search the same three learning rates using three tuning seeds. Corrupt only training labels. Validation labels are clean.

Select each configuration's rate by mean final validation CE. In particular, ordinary CE gets its own rate at every eta, making it the baseline for all epsilons at that eta. No test split is generated during tuning.


In [5]:
started = time.perf_counter()
new_tuning_runs = 0
tuning_data = {seed: prepare_data(seed, include_test=False) for seed in TUNING_SEEDS}

for seed, eta in product(TUNING_SEEDS, ETAS):
    data = tuning_data[seed]
    observed, realized = observed_labels(data, eta)
    for epsilon, lr in product(EPSILONS, LEARNING_RATES):
        record = dict(seed=seed, eta=eta, epsilon=epsilon, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in tuning_cache and tuning_cache[key]["status"] == "ok":
            continue
        try:
            _, validation, seconds = train_one(seed, data, observed, epsilon, lr)
            record.update(
                status="ok", validation_ce=validation["ce"],
                validation_accuracy=validation["accuracy"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        tuning_cache[key] = record
        new_tuning_runs += 1
    write_csv(TUNING_PATH, list(tuning_cache.values()))
    print(f"Tuning seed={seed}, eta={eta:g}: {len(tuning_cache)}/{TOTAL_TUNING_RUNS} records.", flush=True)

tuning_records = list(tuning_cache.values())
tuning_summary = []
selected = {}
for eta, epsilon in product(ETAS, EPSILONS):
    candidates = []
    for lr in LEARNING_RATES:
        runs = [tuning_cache.get((seed, eta, epsilon, lr)) for seed in TUNING_SEEDS]
        eligible = all(row is not None and row["status"] == "ok" for row in runs)
        entry = dict(eta=eta, epsilon=epsilon, lr=lr, eligible=eligible)
        if eligible:
            entry["validation_ce_mean"], entry["validation_ce_std"] = summarize([r["validation_ce"] for r in runs])
            candidates.append(entry)
        tuning_summary.append(entry)
    if not candidates:
        raise RuntimeError(f"No complete valid learning-rate candidate for eta={eta}, epsilon={epsilon}.")
    best = min(candidates, key=lambda row: (row["validation_ce_mean"], row["lr"]))
    selected[(eta, epsilon)] = best["lr"]

frozen_settings = [
    dict(eta=eta, epsilon=epsilon, lr=lr) for (eta, epsilon), lr in selected.items()
]
selection_snapshot = json.dumps(frozen_settings, sort_keys=True)
(RUN_DIR / "selected_settings.json").write_text(json.dumps(frozen_settings, indent=2) + "\n", encoding="utf-8")
write_csv(RUN_DIR / "tuning_summary.csv", tuning_summary)
boundary_count = sum(lr in (min(LEARNING_RATES), max(LEARNING_RATES)) for lr in selected.values())
print(f"Tuning finished in {time.perf_counter()-started:.1f} s; {new_tuning_runs} newly trained runs.")
print(f"Selected rates at tested grid endpoints: {boundary_count}/{len(selected)}. Values outside this finite grid were not tested.")


Tuning seed=0, eta=0: 756/756 records.


Tuning seed=0, eta=0.025: 756/756 records.


Tuning seed=0, eta=0.05: 756/756 records.

Tuning seed=0, eta=0.1: 756/756 records.


Tuning seed=0, eta=0.2: 756/756 records.


Tuning seed=0, eta=0.3: 756/756 records.


... (12 outputs left out)

Tuning seed=2, eta=0.2: 756/756 records.


Tuning seed=2, eta=0.3: 756/756 records.


Tuning seed=2, eta=0.4: 756/756 records.


Tuning finished in 0.2 s; 0 newly trained runs.
Selected rates at tested grid endpoints: 73/84. Values outside this finite grid were not tested.


## 4. Evaluation on matched seeds

Use the same five evaluation seeds as notebook 03: 201 through 205. Clean validation/test labels, data generation, initializations, and corruption masks are identical, allowing the same CE baseline to be used when compatible.

Train smoothing models from scratch at their validation-selected learning rates. Epsilon = 0 is the CE control. Evaluate the final checkpoint on clean test data; test losses do not select smoothing strength, learning rate, or checkpoint.


In [6]:
started = time.perf_counter()
new_evaluation_runs = 0
evaluation_data = {seed: prepare_data(seed, include_test=True) for seed in EVALUATION_SEEDS}

for seed, eta in product(EVALUATION_SEEDS, ETAS):
    data = evaluation_data[seed]
    observed, realized = observed_labels(data, eta)
    for epsilon in EPSILONS:
        lr = selected[(eta, epsilon)]
        record = dict(seed=seed, eta=eta, epsilon=epsilon, lr=lr, realized_corruption=realized)
        key = run_key(record)
        if key in evaluation_cache and evaluation_cache[key]["status"] == "ok":
            continue
        try:
            model, validation, seconds = train_one(seed, data, observed, epsilon, lr)
            test = evaluate(model, data["test"])
            record.update(
                status="ok", validation_ce=validation["ce"],
                test_ce=test["ce"], test_accuracy=test["accuracy"],
                test_confidence=test["confidence"], training_seconds=seconds,
            )
        except (FloatingPointError, RuntimeError) as error:
            record.update(status="failed", error=str(error))
        evaluation_cache[key] = record
        new_evaluation_runs += 1
    write_csv(EVALUATION_PATH, list(evaluation_cache.values()))
    print(f"Evaluation seed={seed}, eta={eta:g}: {len(evaluation_cache)}/{TOTAL_EVALUATION_RUNS} records.", flush=True)

evaluation_records = list(evaluation_cache.values())
assert selection_snapshot == json.dumps([
    dict(eta=eta, epsilon=epsilon, lr=lr) for (eta, epsilon), lr in selected.items()
], sort_keys=True)
print(f"Evaluation finished in {time.perf_counter()-started:.1f} s; {new_evaluation_runs} newly trained runs.")


Evaluation seed=201, eta=0: 420/420 records.


Evaluation seed=201, eta=0.025: 420/420 records.

Evaluation seed=201, eta=0.05: 420/420 records.


Evaluation seed=201, eta=0.1: 420/420 records.

Evaluation seed=201, eta=0.2: 420/420 records.


Evaluation seed=201, eta=0.3: 420/420 records.

... (26 outputs left out)

Evaluation seed=205, eta=0.2: 420/420 records.


Evaluation seed=205, eta=0.3: 420/420 records.


Evaluation seed=205, eta=0.4: 420/420 records.

Evaluation finished in 0.2 s; 0 newly trained runs.


## 5. Paired comparison against ordinary CE

A cell is a mean across the same five paired evaluation seeds. Save both losses, mean gain, relative reduction, sample standard deviation of paired gains, and the number of seeds with gain above the numerical tolerance.

A positive five-seed mean is an observed win in this experiment, not a claim of statistical significance. The epsilon = 0 control column must have exactly zero gain.


In [7]:
paired_records = []
cell_records = []
for eta, epsilon in product(ETAS, EPSILONS):
    baseline_runs, smoothing_runs, gains = [], [], []
    for seed in EVALUATION_SEEDS:
        baseline = evaluation_cache.get((seed, eta, 0.0, selected[(eta, 0.0)]))
        smoothing = evaluation_cache.get((seed, eta, epsilon, selected[(eta, epsilon)]))
        if baseline is None or smoothing is None or baseline["status"] != "ok" or smoothing["status"] != "ok":
            continue
        assert baseline["realized_corruption"] == smoothing["realized_corruption"]
        gain = baseline["test_ce"] - smoothing["test_ce"]
        baseline_runs.append(baseline)
        smoothing_runs.append(smoothing)
        gains.append(gain)
        paired_records.append(dict(
            seed=seed, eta=eta, epsilon=epsilon,
            baseline_lr=baseline["lr"], smoothing_lr=smoothing["lr"],
            baseline_test_ce=baseline["test_ce"], smoothing_test_ce=smoothing["test_ce"],
            gain=gain, win=gain > WIN_ATOL,
        ))
    complete = len(gains) == len(EVALUATION_SEEDS)
    record = dict(
        eta=eta, epsilon=epsilon, complete=complete, paired_seeds=len(gains),
        baseline_lr=selected[(eta, 0.0)], smoothing_lr=selected[(eta, epsilon)],
    )
    if complete:
        baseline_mean = statistics.mean(r["test_ce"] for r in baseline_runs)
        smoothing_mean = statistics.mean(r["test_ce"] for r in smoothing_runs)
        gain_mean, gain_std = summarize(gains)
        record.update(
            baseline_test_ce_mean=baseline_mean, smoothing_test_ce_mean=smoothing_mean,
            gain_mean=gain_mean, gain_std=gain_std,
            relative_gain_percent=100 * gain_mean / baseline_mean,
            seed_wins=sum(gain > WIN_ATOL for gain in gains),
            mean_win=gain_mean > WIN_ATOL,
        )
        if epsilon == 0:
            assert all(gain == 0 for gain in gains)
    cell_records.append(record)

write_csv(RUN_DIR / "paired_results.csv", paired_records)
write_csv(RUN_DIR / "grid_summary.csv", cell_records)
complete_cells = [r for r in cell_records if r["complete"]]
nonzero_cells = [r for r in complete_cells if r["epsilon"] > 0]
winning_cells = [r for r in nonzero_cells if r["mean_win"]]
print(f"Complete cells: {len(complete_cells)}/{len(ETAS)*len(EPSILONS)}.")
print(f"Observed mean wins at epsilon > 0: {len(winning_cells)}/{len(nonzero_cells)}.")
if winning_cells:
    for eta in ETAS:
        wins = [r for r in winning_cells if r["eta"] == eta]
        if wins:
            print(f"eta={eta:g}: {len(wins)} winning cells; epsilons=" + ", ".join(f"{r['epsilon']:g}" for r in wins))
else:
    print("No tested positive epsilon beat its matched CE baseline beyond the numerical tolerance.")


Complete cells: 84/84.


Observed mean wins at epsilon > 0: 3/77.
eta=0: 3 winning cells; epsilons=0.0002, 0.002, 0.005


## 6. Eta versus epsilon: red-white-green comparison grid

- **Green:** positive gain; smoothing lowers ordinary clean test CE.
- **Red:** negative gain; smoothing increases ordinary clean test CE.
- **White:** zero or very small absolute change. Numerical ties are mapped exactly to zero.
- **Saturation:** absolute gain magnitude on one symmetric linear scale centered at zero.
- **Missing:** a cell without all five valid paired seeds is left blank.

Every tested pair occupies an equal-size categorical bin. Cell labels and hover preserve the actual signed mean gain; hover also shows both losses, relative change, paired variation, seed wins, and selected learning rates.

The numerical win rule remains mean gain > 0.000001. Very small observed wins can look almost white because they are small on the common loss scale. The zero-epsilon control is white.


In [8]:
by_cell = {(r["eta"], r["epsilon"]): r for r in cell_records}
color_grid, text_grid, hover_grid = [], [], []
for eta in ETAS:
    colors, texts, details = [], [], []
    for epsilon in EPSILONS:
        record = by_cell[(eta, epsilon)]
        if not record["complete"]:
            colors.append(None)
            texts.append("missing")
            details.append([eta, epsilon, None, None, None, None, None, None, None, None, None])
            continue
        gain = record["gain_mean"]
        # Keep zero and numerical ties white; use actual signed gains elsewhere.
        color_gain = 0.0 if abs(gain) <= WIN_ATOL else gain
        colors.append(color_gain)
        texts.append("tie" if abs(gain) <= WIN_ATOL else f"{gain:+.3f}" if abs(gain) >= 0.001 else f"{gain:+.1e}")
        details.append([
            eta, epsilon, record["baseline_test_ce_mean"], record["smoothing_test_ce_mean"],
            gain, record["relative_gain_percent"], record["gain_std"],
            record["seed_wins"], record["paired_seeds"], record["baseline_lr"], record["smoothing_lr"],
        ])
    color_grid.append(colors)
    text_grid.append(texts)
    hover_grid.append(details)

color_limit = max([abs(v) for row in color_grid for v in row if v is not None] + [WIN_ATOL])
colorscale = [
    [0.0, "#991b1b"],
    [0.5, "#ffffff"],
    [1.0, "#166534"],
]
fig = go.Figure(go.Heatmap(
    x=[f"{epsilon:g}" for epsilon in EPSILONS],
    y=[f"{eta:g}" for eta in ETAS],
    z=color_grid, zmin=-color_limit, zmax=color_limit,
    colorscale=colorscale,
    text=text_grid, texttemplate="%{text}", textfont=dict(size=10, color="#111827"),
    customdata=hover_grid, xgap=2, ygap=2, hoverongaps=False,
    colorbar=dict(title="CE baseline - Smoothing", thickness=18),
    hovertemplate=(
        "eta=%{customdata[0]}<br>epsilon=%{customdata[1]}"
        "<br>Mean baseline test CE=%{customdata[2]:.8f}"
        "<br>Mean Smoothing test CE=%{customdata[3]:.8f}"
        "<br>Mean paired gain=%{customdata[4]:+.8f}"
        "<br>Relative CE reduction=%{customdata[5]:+.4f}%"
        "<br>Paired gain sample SD=%{customdata[6]:.8f}"
        "<br>Seed wins=%{customdata[7]}/%{customdata[8]}"
        "<br>Selected baseline lr=%{customdata[9]}"
        "<br>Selected Smoothing lr=%{customdata[10]}"
        "<extra></extra>"
    ),
))
fig.update_xaxes(title_text="Label-smoothing strength epsilon", type="category", tickangle=-35, side="bottom")
fig.update_yaxes(title_text="Training-label flip probability eta", type="category", autorange="reversed")
fig.update_layout(
    title="Gaussian: did Smoothing beat ordinary CE on clean test CE?",
    height=650, margin=dict(l=90, r=180, t=100, b=120),
    plot_bgcolor="#e5e7eb",
    annotations=[dict(
        x=0, y=1.08, xref="paper", yref="paper", showarrow=False, xanchor="left",
        text="Green: lower CE; red: higher CE; white: near zero. Five matched evaluation seeds.",
    )],
)
fig.show(config={"displaylogo": False, "responsive": True})
fig.write_html(RUN_DIR / "eta_epsilon_heatmap.html", include_plotlyjs=True, full_html=True)


## 7. Interpretation and scope

Green cells identify observed reductions in ordinary clean test CE relative to a separately validation-tuned CE baseline at the same eta. Red cells mean worse CE; white or nearly white cells indicate zero or small changes. Finding no meaningful improvement is valid.

Use magnitude and repeatability together: hover shows mean gain, paired sample standard deviation, and wins among five seeds. Tiny green cells may be practically negligible or sensitive to seed choice. This is an exploratory test grid; a proposed setting should be frozen and checked on fresh seeds before a confirmatory claim.

The Gaussian data, 1,000 training examples, logistic model, 400-step budget, eta grid, tuning seeds, evaluation seeds, and numerical win tolerance match notebook 03. Epsilon values are twice its delta values to align positive-margin penalties. The smoothing and RobustCE losses still differ on negative margins; these grids do not prove one method is universally superior.


In [9]:
failures = [
    {"phase": phase, **record}
    for phase, records in (("tuning", tuning_records), ("evaluation", evaluation_records))
    for record in records if record["status"] != "ok"
]
protocol = {
    **settings, "run_id": fingerprint,
    "test_metric": "ordinary CE on clean held-out labels",
    "gain": "matched CE baseline test CE minus Smoothing test CE",
    "baseline": "epsilon=0, independently validation-tuned lr at each eta",
    "win_rule": f"five-seed mean paired gain > {WIN_ATOL}",
    "heatmap_colors": "continuous red-white-green; zero-centered; numerical ties at zero",
    "zero_smoothing_control": "exact same CE model/record reused; gain must be zero",
    "cache_reuse": REUSE_CACHE,
    "ce_baseline_source": str(baseline_source) if baseline_source is not None else None,
    "smoothing_definition": "soft target (1-epsilon)*y + epsilon/2",
    "grid_relation_to_notebook_03": "epsilon=2*delta",
    "unique_tuning_runs": TOTAL_TUNING_RUNS,
    "unique_evaluation_runs": TOTAL_EVALUATION_RUNS,
    "complete_cells": len(complete_cells),
    "positive_epsilon_cells": len(nonzero_cells),
    "observed_winning_cells": len(winning_cells),
    "failed_runs": len(failures),
    "scope": "exploratory eta/epsilon test grid; no test-based deployment selection",
}
(RUN_DIR / "protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
(RUN_DIR / "failures.json").write_text(json.dumps(failures, indent=2) + "\n", encoding="utf-8")
assert len(tuning_records) == TOTAL_TUNING_RUNS
assert len(evaluation_records) == TOTAL_EVALUATION_RUNS
assert len(cell_records) == len(ETAS) * len(EPSILONS)
assert len([r for r in evaluation_records if r["epsilon"] == 0]) == len(ETAS) * len(EVALUATION_SEEDS)
print(
    f"Saved {len(cell_records)} cells from {len(tuning_records)} tuning and "
    f"{len(evaluation_records)} evaluation records. Failed runs: {len(failures)}."
)
print(f"Artifacts and interactive HTML: {RUN_DIR}")


Saved 84 cells from 756 tuning and 420 evaluation records. Failed runs: 0.
Artifacts and interactive HTML: C:\Users\shich\Src\hw1opt\results\04_gaussian_eta_epsilon\5f24bd06b1f030c3
